In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta


In [ ]:
categories=pd.read_csv('../datasets/domens_categories.csv')
ranks=pd.read_csv('../datasets/ranked_top1000000.csv')
secondary_prices=pd.read_csv('../datasets/secondary_prices.csv')

Датасет с целевой переменной

In [ ]:
ton_domens=pd.read_csv('../raw_datasets/ton_domains_history.csv').drop(['tx_hash','sale_type'],axis=1)

убираем .ton

In [22]:
ton_domens['domain_name']=ton_domens['domain_name'].map(lambda x: x.split('.')[0])

Так как это список продаж, один и тот же домен может встречаться несколько раз. Избавляемся от дубликатов, а актуальную цену берём по последней продаже 

In [23]:
ton_domens['tx_time'] = pd.to_datetime(ton_domens['tx_time'], unit='s')
ton_domens.sort_values(by='tx_time', inplace=True)
ton_domens = ton_domens.groupby(by='domain_name', as_index=False).agg(ton_price=('price_ton', 'last'))
ton_domens = ton_domens.rename(columns={'domain_name': 'domain'})

Собираем итоговый датасет

In [24]:
ton_dataset = ton_domens.merge(categories, on='domain', how='left')
len(ton_dataset[ton_dataset['category'].notna()])

3121

Всего 3121 доменов размеченных по категориям

In [25]:
ton_dataset=ton_dataset.merge(ranks, on='domain', how='left')

In [26]:
secondary_prices.rename(columns={'domain_name':'domain'}, inplace=True)
main_dataset = secondary_prices.merge(ton_dataset, on='domain', how='left')
main_dataset

,domain,price,eth_reg_price,eth_sales_count,eth_sales_volume_usd,eth_last_sale_price_usd,usd_price,sol_sales_count,sol_sales_volume_usd,sol_last_sale_price_usd,ton_price,category,rank
0,rilxxlir,0.007863,17.22,NaN,NaN,NaN,19.387756,NaN,NaN,NaN,NaN,NaN,NaN
1,hansberry,0.003712,8.13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,lalegend,0.002886,6.32,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,peterwagner,0.001976,4.33,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,stateland,0.000544,1.19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3490611,gangster-all-star,0.000481,1.05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3490612,chain-feeds,0.002754,6.03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3490613,68435,0.009944,21.78,1.0,38.41376,38.41376,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3490614,prismai-deployer,0.001257,2.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
dataset_with_ton = main_dataset[main_dataset["ton_price"].notna()]
dataset_without_ton = main_dataset[main_dataset["ton_price"].isna()]
dataset_with_ton.to_csv("../datasets/dataset_main.csv")
dataset_without_ton.to_csv("../datasets/dataset_without_target_value.csv")
dataset_with_ton

Мы получили два датасета, один с без размеченной целевой переменной, другой без нее. Первый датасет может быть использован для обучения и проверки модели, а второй будет использоваться для отображения потенциальной цены продажи нового домена на сайте webdom.market